In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")


In [3]:
import os

os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## Messagebased summarization

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


In [11]:
# run with thread id
config = {'configurable':{'thread_id':"test-1"}}

In [12]:
# Alternative test data

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke(
        {"messages": [HumanMessage(content=q)]},
        config
    )

    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")
    

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a48afd5f-880f-4c07-bc84-3ac2b0379910'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EjQKMgERTTIP0yqOYvmoa1Gh9mcOrltf01j3GaI/D1snuNymzZTSsm/et1ACfinxj4Amk36+'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f8bbb-ba72-74b0-876d-d5062240ee4f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a48afd5f-880f-4c07-bc84-3ac2b0379910'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'EjQKMgERTTIP0yqOYvmoa1Gh9mcOrltf01j3GaI/D1snuNymzZTSsm/et1ACfinxj4Amk36+'}}

### Token Size

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""

    return f"""Hotels in {city}:
1. Grand Hotel - 5 star, $350/night, spa, pool, gym
2. City Inn - 4 star, $180/night, business center
3. Budget Stay - 3 star, $75/night, free wifi"""